# Django, URL Routing

## Introduction to URL Routing
---

In this lesson, you'll learn how **URL routing** works in Django.

<br>

URL routing is the mechanism that maps URL addresses to the code that handles them - your views.

To put it simply, URL routing is something in Django like a **receptionist** in a big hotel or a signpost at an intersection.

It just connects the right dots.

---

1. [URLconf Basics](#URLconf-Basics),
    - [What is URLconf](#What-is-URLconf),
    - [The path() Function](#The-path()-Function),
2. [URL Parameters](#URL-Parameters),
    - [Path Converters](#Path-Converters),
    - [Capturing Values from URLs](#Capturing-Values-from-URLs),
3. [Modular URLs with include()](#Modular-URLs-with-include()),
    - [App-Level URL Configuration](#App-Level-URL-Configuration),
    - [URL Prefixes](#URL-Prefixes),
4. [URL Naming and reverse()](#URL-Naming-and-reverse()),
    - [Why Name URLs](#Why-Name-URLs),
    - [Using reverse() in Code](#Using-reverse()-in-Code),
    - [URL Namespaces](#URL-Namespaces),
5. [🧠 EXERCISE 🧠, Create URL Patterns](#🧠-EXERCISE-🧠,-Create-URL-Patterns).

<br>

## URLconf Basics

---

### What is URLconf

A **URLconf** (URL configuration) is a mapping between URL patterns and view functions.

Think of it as a **table of contents** for your website - it tells Django which code to run for each URL.

<br>

| URL | View Function | What it does |
|-----|---------------|-------------|
| `/` | `home` | Shows homepage |
| `/blogs/` | `blog_list` | Lists all blogs |
| `/blogs/5/` | `blog_detail` | Shows blogs with ID 5 |
| `/about/` | `about` | Shows about page |

<br>

The URLconf is defined in `urls.py` files using a list called `urlpatterns`.

<br>

### The `path()` Function

---

It's primary way to define URL patterns in Django.

<br>

**Syntax:**

```python
path(route, view, kwargs=None, name=None)
```

<br>

| Parameter | Description |
|-----------|-------------|
| `route` | URL pattern to match (e.g., `'books/'`) |
| `view` | View function or class to call |
| `kwargs` | Additional arguments to pass to the view |
| `name` | Name for the URL pattern (for reverse lookups) |

In [ ]:
# Example: Basic urls.py structure

from django.urls import path
from blog import views  # Note: Dot annotation is optional, relative import statement

urlpatterns = [
    path('', views.home, name='home'),                  # Matches: /
    path('blogs/', views.blog_list, name='blog_list'),  # Matches: /blogs/
]

<br>

**Key points about URL routes:**

- Routes **don't start with a slash** - Django adds it automatically
- Empty string `''` matches the root URL `/`
- Django tries patterns **in order** - first match wins
- Trailing slashes are **recommended** for consistency

In [ ]:
# Example: Simple view functions for the URLs above
# blog/views.py

from django.http import HttpResponse

def home(request):
    return HttpResponse("Welcome to the Blog editor!")

def blog_list(request):
    return HttpResponse("Here's a list of all blogs.")

def about(request):
    return HttpResponse("About our blog editor.")

<br>

## URL Parameters

---

### Path Converters

Often you need **dynamic URLs** that capture values - like a book ID or a username.

Django uses **path converters** to capture parts of the URL and pass them to your view.

<br>

**Syntax:** `<converter:name>`

| Converter | Matches | Example |
|-----------|---------|--------|
| `int` | Positive integers | `<int:pk>` → `1`, `42`, `999` |
| `str` | Non-empty string (excludes `/`) | `<str:title>` → `hello`, `my-blog` |
| `slug` | ASCII letters, numbers, hyphens, underscores | `<slug:slug>` → `my-first-post` |
| `uuid` | UUID format | `<uuid:id>` → `075194d3-...` |
| `path` | Any string including `/` | `<path:file>` → `docs/intro.txt` |

In [ ]:
# Example: URL patterns with parameters

from django.urls import path
from . import views

urlpatterns = [
    # Integer parameter - for database IDs
    path('/<int:pk>', views.blog_detail, name='blog_detail'),
    # Matches: /blogs/1/, /blogs/42/, /blogs/999/
    
    # Slug parameter - for SEO-friendly URLs
    path('<slug:slug>/', views.blog_by_slug, name='blog_by_slug'),
    # Matches: /blogs/django-for-beginners/, /blogs/python-crash-course/
    
    # Multiple parameters
    path('<int:year>/<int:month>/', views.blogs_archive, name='blogs_archive'),
    # Matches: /blogs/2024/03/, /blogs/2023/12/
]

<br>

### Capturing Values from URLs

---

When a URL matches, the captured values are passed to your view as **keyword arguments**.

In [ ]:
# Example: View functions receiving URL parameters
from django.http import HttpRequest, JsonResponse
from django.shortcuts import get_object_or_404

from blog.models import Blog


def blog_list(request: HttpRequest):
    blogs = Blog.objects.all()
    return JsonResponse(list(blogs.values()), safe=False)

def blog_detail(request: HttpRequest, pk: int):
    blog = Blog.objects.get(pk=pk)
    return JsonResponse({
        'id': blog.id,
        'title': blog.title,
        'author': blog.author,
        'published_date': blog.published_date.isoformat()
    })

<br>

**Type conversion happens automatically:**

| URL | Converter | Received Value |
|-----|-----------|----------------|
| `/books/42/` | `<int:pk>` | `pk=42` (integer) |
| `/books/hello/` | `<str:title>` | `title="hello"` (string) |
| `/books/my-book/` | `<slug:slug>` | `slug="my-book"` (string) |

In [ ]:
# Example: Real-world blog detail view

from django.http import HttpResponse, Http404
from blog.models import Blog

def blog_detail(request, pk: int):
    """Display blog details or return 404."""
    blog = get_object_or_404(Blog, pk=pk)

    return JsonResponse({
        'id': blog.id,
        'title': blog.title,
        'author': blog.author,
        'published_date': blog.published_date.isoformat()
    })

# URL: /blogs/1/  →  "Django for Beginners by William Vincent"
# URL: /blogs/99/ →  404 Not Found

<br>

## Modular URLs with `include()`

---

### App-Level URL Configuration

As your project grows, putting all URLs in one file becomes messy.

Django's `include()` function lets you **split URLs by app**.

<br>

**The pattern:**

1. Create `urls.py` in each app
2. Use `include()` in the project's `urls.py` to reference them

In [ ]:
# blog/urls.py - App-specific URLs

from django.urls import path
from blog import views

app_name = 'blog'  # Namespace for this app's URLs

urlpatterns = [
    path('', views.blog_list, name='blog_list'),
    path('<int:pk>/', views.blog_detail, name='blog_detail'),
]

In [ ]:
# mysite/urls.py - Project-level URLs

from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('blogs/', include('blog.urls')),    # All /blogs/... URLs
    # TODO
]

<br>

**Result:**

| Full URL | Handled by |
|----------|------------|
| `/blog/` | `blog.views.blog_list` |
| `/blogs/5/` | `blog.views.blog_detail` |
| E.g. `/accounts/login/` | `accounts.views.login` |

<br>

### URL Prefixes

---

The first argument to `path()` before `include()` becomes a **prefix** for all included URLs.

`path()` is basically a building block.

It's a precise instruction that says, **"When the user types this address into the browser, run this function".**

<br>

It's a traffic sign: **"This is the way to the Shopping District. When you get there, look at their local map to find the individual stores."**.

In [ ]:
# Example: Different prefixes for the same app URLs

from django.urls import path, include

urlpatterns = [
    # Public blog
    path('blogs/', include('blog.urls')),
    # Matches: /blogs/, /blogs/5/, etc.
    
    # Staff preview (same views, different URL prefix)
    path('preview/blogs/', include('blog.urls')),
    # Matches: /preview/blogs/, /preview/blogs/5/, etc.
]

<br>

**Best practice:** Keep your project's `urls.py` clean - only use `include()` to delegate to apps:

```python
# GOOD: Project urls.py with includes only
urlpatterns = [
    path('admin/', admin.site.urls),
    # Prebuilt path^
    path('blogs/', include('blog.urls')),
    path('accounts/', include('accounts.urls')),
]

# AVOID: Mixing includes with direct views
urlpatterns = [
    path('admin/', admin.site.urls),
    path('', views.home),  # Direct view in project urls
    path('blogs/', include('blog.urls')),
    path('contact/', views.contact),  # Another direct view
]
```

<br>

## URL Naming and `reverse()`

---

### Why Name URLs

Hardcoding URLs in your code is **fragile**. If you change a URL pattern, you have to update it everywhere.

<br>

**The problem:**

In [ ]:
# BAD: Hardcoded URLs

def some_view(request):
    # If URL changes from /blogs/ to /catalog/, this breaks!
    return redirect('/blogs/5/')  

# In template (also BAD):
# <a href="/books/5/">View book</a>

<br>

**The solution:** Name your URLs and use `reverse()` to generate them.

In [ ]:
# urls.py - Name your URLs

from django.urls import path
from . import views

urlpatterns = [
    path('', views.blog_list, name='blog_list'),
    path('<int:pk>/', views.blog_detail, name='blog_detail'),
    #                                    ^^^^^^^^^^^^^^^^^
    #                                    URL name for reverse lookup
]

`reverse()` is a function that creates URL address based on the given name.

<br>

### Using `reverse()` in Code

---

The `reverse()` function generates a URL from its name.

In [ ]:
# GOOD: Using reverse() in views

from django.urls import reverse
from django.shortcuts import redirect

def some_view(request):
    # Generate URL dynamically - if URL pattern changes, this still works!
    url = reverse('blog_detail', kwargs={'pk': 5})
    # url = '/blogs/5/'
    return redirect(url)

# Even simpler with redirect shortcut:
def another_view(request):
    return redirect('blog_detail', pk=5)

`redirect()` just terminates your function and tells the browser: **"Go to this page!"**

In [ ]:
# Example: reverse() with different URL patterns

from django.urls import reverse

# Simple URL (no parameters)
url1 = reverse('blog:blog_list')
# Result: '/blogs/'

# URL with one parameter
url2 = reverse('blog:blog_detail', kwargs={'id': 42})
# Result: '/blogs/42/'

# Alternative syntax with args (positional)
url3 = reverse('blog:blog_detail', args=[42])
# Result: '/blogs/42/'

<br>

**In templates**, use the `{% url %}` tag instead of `reverse()`:

In [ ]:
# Template examples (not actual Python, just for illustration)
<!-- Simple URL -->
<a href="{% url 'blog_list' %}">All Blogs</a>

<!-- URL with parameter -->
<a href="{% url 'blog_detail' pk=blog.pk %}">{{ blog.title }}</a>

<!-- URL with multiple parameters -->
<a href="{% url 'blogs_archive' year=2024 month=3 %}">March 2024</a>

<br>

### URL Namespaces

---

When you have multiple apps with similar URL names (like `detail` or `list`), **namespaces** prevent conflicts.

<br>

**Define a namespace** with `app_name` in your app's `urls.py`:

In [ ]:
# blog/urls.py

from django.urls import path
from . import views

app_name = 'blog'  # This is the namespace

urlpatterns = [
    path('', views.blog_list, name='list'),
    path('<int:pk>/', views.blog_detail, name='detail'),
]

In [ ]:
# accounts/urls.py

from django.urls import path
from . import views

app_name = 'accounts'  # Different namespace

urlpatterns = [
    path('', views.user_list, name='list'),      # Same name, different namespace
    path('<int:pk>/', views.user_detail, name='detail'),
]

In [ ]:
# blog/views.py - Using namespaced URLs with reverse()

from django.urls import reverse
from django.shortcuts import redirect

def some_view(request):
    # Use namespace:name syntax to avoid conflicts between apps
    blog_url = reverse('blog:detail', kwargs={'pk': 5})
    # Result: '/blogs/5/'
    
    # BUT RATHER INSIDE ACCOUNTS APP
    accounts_url = reverse('accounts:detail', kwargs={'pk': 5})
    # Result: '/accounts/5/'
    
    return redirect(blog_url)

In [ ]:
# Template with namespaced URLs (in templates, use {% url %} instead of reverse())
<!-- Blog detail -->
<a href="{% url 'blog:detail' pk=blog.pk %}">View Blog</a>

<!-- Accounts user detail -->
<a href="{% url 'accounts:detail' pk=user.pk %}">View Profile</a>

<br>

## Common URL Patterns

---

Here are some common URL patterns you'll encounter:

In [ ]:
# blog/urls.py - CRUD URL patterns

from django.urls import path
from . import views

app_name = 'blog'

urlpatterns = [
    path('', views.blog_list, name='blog_list'),           # GET /blogs/
    path('new/', views.blog_create, name='blog_create'),   # GET + POST /blogs/new/
    path('<int:pk>/', views.blog_detail, name='blog_detail'),         # GET /blogs/5/
    path('<int:pk>/edit/', views.blog_update, name='blog_update'),    # GET + POST /blogs/5/edit/
    path('<int:pk>/delete/', views.blog_delete, name='blog_delete'),  # GET + POST /blogs/5/delete/
]

In [ ]:
# blog/views.py - CRUD views (simplified with HttpResponse for demo)

from django.http import HttpResponse, Http404, JsonResponse
from blog.models import Blog

def blog_list(request):
    blogs = Blog.objects.all()
    return JsonResponse(list(blogs.values()), safe=False)

def blog_detail(request, pk: int):
    blog = get_object_or_404(Blog, pk=pk)

    return JsonResponse({
        'id': blog.id,
        'title': blog.title,
        'author': blog.author,
        'published_date': blog.published_date.isoformat()
    })

def blog_update(request, pk: int):
    blog = get_object_or_404(Blog, pk=pk)

    if request.method == 'POST':
        return HttpResponse(f"Blog {pk} updated!")

def blog_delete(request, pk: int):
    blog = get_object_or_404(Blog, pk=pk)

    if request.method == 'POST':
        return HttpResponse(f"Blog {pk} deleted!")

<br>

> **Note:** This is a simplified demo using `HttpResponse`. In a real project, views would render templates or return JSON.
>
> If you're building a **REST API** (for mobile apps, frontend frameworks like React, etc.), the same URL structure applies — but views return `JsonResponse` instead of HTML, and you typically use `GET`/`POST`/`PUT`/`DELETE` HTTP methods explicitly. That's covered in a later lesson.

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **URLconf** | `urlpatterns` list in `urls.py` - maps URLs to views |
| **path()** | `path(route, view, name=...)` - defines URL pattern |
| **Converters** | `<int:pk>`, `<str:name>`, `<slug:slug>` - capture URL parts |
| **include()** | Delegate URLs to app-specific `urls.py` |
| **URL names** | Always name URLs for `reverse()` lookups |
| **reverse()** | Generate URLs from names - avoids hardcoding |
| **Namespaces** | `app_name` prevents URL name conflicts |

Using the `blog` app from previous lessons:

1. Create URL patterns in `blog/urls.py` for:
   - List all blogs at `/` (name: `blog_list`)
   - Blog detail at `/<int:pk>/` (name: `blog_detail`)
   - Blogs by author at `/author/<int:author_name>/` (name: `blogs_by_author`)
2. Set `app_name = 'blog'` for namespacing
3. Create simple view functions in `blog/views.py` that return `HttpResponse`
4. Include blog URLs in the project's `urls.py` at `/blogs/`
5. Test by running the server and visiting the URLs

<details>
    <summary>▶️ Solution</summary>
    
**1-2. Edit `blog/urls.py`:**

```python
from django.urls import path
from . import views

app_name = 'blog'

urlpatterns = [
    path('', views.blog_list, name='blog_list'),
    path('<int:pk>/', views.blog_detail, name='blog_detail'),
    path('author/<int:author_name>/', views.blogs_by_author, name='blogs_by_author'),
]
```

**3. Edit `blog/views.py`:**

```python
from django.http import HttpResponse

def blog_list(request):
    return HttpResponse("List of all blogs")

def blog_detail(request, pk: int):
    return HttpResponse(f"Blog detail for ID: {pk}")

def blogs_by_author(request, author_id: int):
    return HttpResponse(f"Blogs by author ID: {author_id}")
```

**4. Edit `mysite/urls.py`:**

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('blogs/', include('blog.urls')),
]
```

**5. Test:**

```bash
python manage.py runserver
```

Visit:
- `http://127.0.0.1:8000/blogs/` → "List of all blogs"
- `http://127.0.0.1:8000/blogs/5/` → "Blog detail for ID: 5"
- `http://127.0.0.1:8000/blogs/author/3/` → "Blogs by author ID: 3"
</details>

<details>
    <summary>▶️ Solution</summary>
    
**1-2. Edit `catalog/urls.py`:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('', views.book_list, name='book_list'),
    path('<int:pk>/', views.book_detail, name='book_detail'),
    path('author/<int:author_id>/', views.books_by_author, name='books_by_author'),
]
```

**3. Edit `catalog/views.py`:**

```python
from django.http import HttpResponse

def book_list(request):
    return HttpResponse("List of all books")

def book_detail(request, pk: int):
    return HttpResponse(f"Book detail for ID: {pk}")

def books_by_author(request, author_id: int):
    return HttpResponse(f"Books by author ID: {author_id}")
```

**4. Edit `bookstore_project/urls.py`:**

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('books/', include('catalog.urls')),
]
```

**5. Test:**

```bash
python manage.py runserver
```

Visit:
- `http://127.0.0.1:8000/books/` → "List of all books"
- `http://127.0.0.1:8000/books/5/` → "Book detail for ID: 5"
- `http://127.0.0.1:8000/books/author/3/` → "Books by author ID: 3"
</details>

Practice using `reverse()` to generate URLs:

1. Open the Django shell: `python manage.py shell`
2. Import `reverse`: `from django.urls import reverse`
3. Generate URLs for:
   - The blog list
   - Blog detail for ID 42
   - Blogs by author with ID 7
4. Create a view that redirects from `/` to the blog list

<details>
    <summary>▶️ Solution</summary>
    
**1-3. In Django shell:**

```python
>>> from django.urls import reverse

>>> reverse('blog:blog_list')
'/blogs/'

>>> reverse('blog:blog_detail', kwargs={'pk': 42})
'/blogs/42/'

>>> reverse('blog:blogs_by_author', kwargs={'author_id': 7})
'/blogs/author/7/'

>>> exit()
```

**4. Add redirect view to `blog/views.py`:**

```python
from django.shortcuts import redirect

def home(request):
    return redirect('blog:blog_list')
```

**Add to `mysite/urls.py`:**

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('pages.urls')),
    path('blogs/', include('blog.urls')),
]
```

Now visiting `http://127.0.0.1:8000/` redirects to `/blogs/`.
</details>

---